<a href="https://colab.research.google.com/github/JONNY-ME/multilingual-health-qa/blob/main/04_aya_23_8b_zero_shot_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aya-23-8B Zero-Shot Baseline

This notebook runs an end-to-end zero-shot baseline using `CohereForAI/aya-23-8B` for multilingual health question answering.

The workflow is baseline-only:

- no fine-tuning
- no 4-bit quantization
- local ROUGE-1 and ROUGE-L validation
- checkpointed inference
- strict prediction CSV generation

Recommended runtime: Colab Pro+ GPU with A100, L4, or another high-memory GPU.

## 1. Colab Runtime Setup

Run this notebook in a GPU runtime. This Aya baseline intentionally avoids 4-bit quantization, so it loads the 8B model in automatic precision with `device_map="auto"`.

In [1]:
# Check GPU. This should show an NVIDIA GPU in Colab.
!nvidia-smi

# Install uv first, then install Python dependencies through uv.
# Aya-23 does not need the bleeding-edge Transformers source build used for Gemma 4.
# We also remove torchvision because some Colab runtimes ship a broken torchvision::nms build,
# which can break text-only model loading through Transformers image utilities.
!pip -q install uv
!uv pip install --system -q --upgrade "transformers>=4.51.0" accelerate torch pandas numpy tqdm rouge-score sentencepiece huggingface_hub gdown safetensors protobuf
!uv pip uninstall --system -q -y torchvision || true

import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

import transformers
print("Transformers version:", transformers.__version__)
print("Setup complete. Restart the session after this cell, then run from the top.")

Mon May 18 21:26:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
import gc
import os
import random
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 240)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Data Download and Loading

The notebook downloads the CSV files into `/content/multilingual-health-question-answering/data` when running in Colab. If the files already exist, the download step is skipped.

In [2]:
import gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1gqWeNhgZGqqJtemvn9a1pMuEjVmi2s0t?usp=sharing"
COLAB_PROJECT_DIR = Path("/content/multilingual-health-question-answering")
COLAB_DATA_DIR = COLAB_PROJECT_DIR / "data"
EXPECTED_DATA_FILES = ["Train.csv", "Val.csv", "Test.csv", "SampleSubmission.csv"]

COLAB_DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_before = [name for name in EXPECTED_DATA_FILES if not (COLAB_DATA_DIR / name).exists()]
if missing_before:
    print("Downloading missing data files:", missing_before)
    gdown.download_folder(
        url=DRIVE_FOLDER_URL,
        output=str(COLAB_DATA_DIR),
        quiet=False,
        use_cookies=False,
    )
else:
    print("All expected data files already exist. Skipping download.")

for filename in EXPECTED_DATA_FILES:
    target = COLAB_DATA_DIR / filename
    if target.exists():
        continue
    matches = [path for path in COLAB_PROJECT_DIR.rglob(filename) if path.is_file()]
    if matches:
        shutil.copy2(matches[0], target)

missing_after = [name for name in EXPECTED_DATA_FILES if not (COLAB_DATA_DIR / name).exists()]
if missing_after:
    raise FileNotFoundError(f"Could not find expected data files after download: {missing_after}")

for filename in EXPECTED_DATA_FILES:
    path = COLAB_DATA_DIR / filename
    print(f"- {path} ({path.stat().st_size / 1024**2:.2f} MB)")

All expected data files already exist. Skipping download.
- /content/multilingual-health-question-answering/data/Train.csv (18.27 MB)
- /content/multilingual-health-question-answering/data/Val.csv (4.26 MB)
- /content/multilingual-health-question-answering/data/Test.csv (0.34 MB)
- /content/multilingual-health-question-answering/data/SampleSubmission.csv (0.24 MB)


In [3]:
PROJECT_DIR_CANDIDATES = [
    Path("/content/multilingual-health-question-answering"),
    Path("/content/drive/MyDrive/multilingual-health-question-answering"),
    Path.cwd(),
]

PROJECT_DIR = None
for candidate in PROJECT_DIR_CANDIDATES:
    if (candidate / "data" / "Train.csv").exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find data/Train.csv. Set PROJECT_DIR manually.")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "Train.csv"
VAL_PATH = DATA_DIR / "Val.csv"
TEST_PATH = DATA_DIR / "Test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "SampleSubmission.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub_df = pd.read_csv(SAMPLE_SUB_PATH)

REQUIRED_COLUMNS = {
    "train": {"ID", "input", "output", "subset"},
    "val": {"ID", "input", "output", "subset"},
    "test": {"ID", "input", "subset"},
    "sample_submission": {"ID", "TargetRLF1", "TargetR1F1", "TargetLLM"},
}

for name, df in {
    "train": train_df,
    "val": val_df,
    "test": test_df,
    "sample_submission": sample_sub_df,
}.items():
    missing = REQUIRED_COLUMNS[name] - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

assert test_df["ID"].tolist() == sample_sub_df["ID"].tolist(), "Test IDs do not match sample submission order."

print("PROJECT_DIR:", PROJECT_DIR)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
display(train_df["subset"].value_counts().sort_index().to_frame("train_rows"))

PROJECT_DIR: /content/multilingual-health-question-answering
Train: (29815, 4)
Validation: (6686, 4)
Test: (2618, 3)


,train_rows
subset,
Aka_Gha,4455
Amh_Eth,1845
Eng_Eth,3915
Eng_Gha,4443
Eng_Ken,2080
Eng_Uga,7624
Lug_Uga,3383
Swa_Ken,2070


## 3. Aya-23-8B Model Loading

This model is loaded without 4-bit quantization. If Colab memory is insufficient, use a higher-memory GPU rather than enabling 4-bit for this baseline.

In [4]:
# Optional Hugging Face login. Run only if model download fails due to authentication.
# from huggingface_hub import notebook_login
# notebook_login()

MODEL_ID = "CohereForAI/aya-23-8B"
USE_4BIT = False

print("MODEL_ID:", MODEL_ID)
print("USE_4BIT:", USE_4BIT)

MODEL_ID: CohereForAI/aya-23-8B
USE_4BIT: False


In [5]:
# Keep torchvision disabled for this text-only notebook before importing Transformers model classes.
# The setup cell installs a recent Transformers version and removes torchvision to avoid
# Colab's common `torchvision::nms does not exist` import error.
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

from transformers import AutoModelForCausalLM, AutoTokenizer


def clear_gpu_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


clear_gpu_memory()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model.eval()

print("Model loaded.")
if hasattr(model, "hf_device_map"):
    print("Device map:", model.hf_device_map)
if torch.cuda.is_available():
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
    print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))

config.json:   0%|          | 0.00/640 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Model loaded.
Allocated GB: 14.95
Reserved GB: 15.24


In [6]:
SUBSET_HINTS = {
    "Aka_Gha": "Akan/Twi as used in Ghana",
    "Lug_Uga": "Luganda as used in Uganda",
    "Swa_Ken": "Kiswahili as used in Kenya",
    "Amh_Eth": "Amharic as used in Ethiopia",
    "Eng_Uga": "English as used in Uganda",
    "Eng_Gha": "English as used in Ghana",
    "Eng_Ken": "English as used in Kenya",
    "Eng_Eth": "English as used in Ethiopia",
}

SYSTEM_PROMPT = """You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.
Answer accurately, clearly, and respectfully.
Do not invent medical facts, drug dosages, or legal requirements.
If the question is in a non-English African language, answer in that same language.
Return only the final answer text. Do not add labels, markdown, explanations, translations, or citations."""


def build_user_prompt(question: str, subset: str) -> str:
    language_hint = SUBSET_HINTS.get(subset, subset)
    return f"""Subset/language cue: {subset} ({language_hint})

Question:
{question}

Write a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer."""


def build_messages(question: str, subset: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(question, subset)},
    ]


def render_prompt(question: str, subset: str) -> str:
    messages = build_messages(question, subset)
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

print(render_prompt(val_df.iloc[0]["input"], val_df.iloc[0]["subset"])[:1000])

<BOS_TOKEN><|START_OF_TURN_TOKEN|><|SYSTEM_TOKEN|>You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.
Answer accurately, clearly, and respectfully.
Do not invent medical facts, drug dosages, or legal requirements.
If the question is in a non-English African language, answer in that same language.
Return only the final answer text. Do not add labels, markdown, explanations, translations, or citations.<|END_OF_TURN_TOKEN|><|START_OF_TURN_TOKEN|><|USER_TOKEN|>Subset/language cue: Aka_Gha (Akan/Twi as used in Ghana)

Question:
Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, atubrafo, anaa wɔn a wɔte beae a ntawantawa wɔ mu?

Write a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer.<|END_OF_TURN_TOKEN|><|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>


In [7]:
THINKING_PATTERNS = [
    r"<\|channel\>thought.*?(?=<\|channel\>|$)",
    r"<think>.*?</think>",
]
SPECIAL_TOKEN_PATTERN = re.compile(r"<\|[^>]+\|>|</?s>|<pad>|<bos>|<eos>")
ROLE_PREFIX_PATTERN = re.compile(
    r"^(assistant|model|answer|final answer|response)\s*[:\-]\s*",
    flags=re.IGNORECASE,
)


def clean_generation(text: str) -> str:
    if text is None:
        return ""
    cleaned = str(text)
    for pattern in THINKING_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.DOTALL | re.IGNORECASE)
    cleaned = SPECIAL_TOKEN_PATTERN.sub(" ", cleaned)
    cleaned = cleaned.replace("```", " ")
    cleaned = ROLE_PREFIX_PATTERN.sub("", cleaned.strip()).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip(' \"\'')


@torch.inference_mode()
def generate_answer(
    question: str,
    subset: str,
    max_new_tokens: int = 220,
    temperature: float = 0.0,
    top_p: float = 0.95,
    repetition_penalty: float = 1.05,
) -> str:
    prompt = render_prompt(question, subset)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if temperature and temperature > 0:
        generation_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": top_p})
    else:
        generation_kwargs["do_sample"] = False

    outputs = model.generate(**inputs, **generation_kwargs)
    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=False)
    return clean_generation(raw)

print("Generation utilities ready.")

Generation utilities ready.


In [8]:
def run_generation_with_checkpoints(
    df: pd.DataFrame,
    output_path: Path,
    checkpoint_every: int = 25,
    max_new_tokens: int = 220,
    temperature: float = 0.0,
    top_p: float = 0.95,
    overwrite: bool = False,
) -> pd.DataFrame:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        existing = pd.read_csv(output_path)
        pred_map = dict(zip(existing["ID"], existing["prediction"].fillna(""))) if {"ID", "prediction"}.issubset(existing.columns) else {}
    else:
        pred_map = {}

    rows = []
    start_time = time.time()
    for _, row in tqdm(df.iterrows(), total=len(df)):
        row_id = row["ID"]
        if row_id in pred_map and str(pred_map[row_id]).strip():
            prediction = pred_map[row_id]
        else:
            prediction = generate_answer(
                row["input"],
                row["subset"],
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )
            pred_map[row_id] = prediction

        rows.append({
            "ID": row_id,
            "subset": row["subset"],
            "input": row["input"],
            "prediction": prediction,
        })

        if len(rows) % checkpoint_every == 0:
            pd.DataFrame(rows).to_csv(output_path, index=False)

    result_df = pd.DataFrame(rows)
    result_df.to_csv(output_path, index=False)
    print(f"Saved {len(result_df)} predictions to {output_path}")
    print(f"Elapsed: {(time.time() - start_time) / 60:.1f} minutes")
    return result_df

print("Checkpointed inference ready.")

Checkpointed inference ready.


## 4. Validation Scoring

This computes the deterministic ROUGE-1 F1 and ROUGE-L F1 components locally. The LLM judge component is tracked from public leaderboard runs separately.

In [9]:
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)


def score_pair(reference: str, prediction: str) -> dict[str, float]:
    reference = "" if pd.isna(reference) else str(reference)
    prediction = "" if pd.isna(prediction) else str(prediction)
    scores = rouge.score(reference, prediction)
    return {
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rougeL_f1": scores["rougeL"].fmeasure,
    }


def score_predictions(pred_df: pd.DataFrame, reference_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    merged = pred_df.merge(
        reference_df[["ID", "output"]],
        on="ID",
        how="left",
        validate="one_to_one",
    )
    score_rows = [score_pair(ref, pred) for ref, pred in zip(merged["output"], merged["prediction"])]
    scored = pd.concat([merged.reset_index(drop=True), pd.DataFrame(score_rows)], axis=1)
    scored["local_rouge_weighted"] = 0.37 * scored["rouge1_f1"] + 0.37 * scored["rougeL_f1"]

    summary = scored.groupby("subset").agg(
        rows=("ID", "count"),
        rouge1_f1=("rouge1_f1", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        local_rouge_weighted=("local_rouge_weighted", "mean"),
        pred_words=("prediction", lambda s: np.mean([len(str(x).split()) for x in s])),
        ref_words=("output", lambda s: np.mean([len(str(x).split()) for x in s])),
    ).reset_index()

    overall = pd.DataFrame([{
        "subset": "OVERALL",
        "rows": len(scored),
        "rouge1_f1": scored["rouge1_f1"].mean(),
        "rougeL_f1": scored["rougeL_f1"].mean(),
        "local_rouge_weighted": scored["local_rouge_weighted"].mean(),
        "pred_words": np.mean([len(str(x).split()) for x in scored["prediction"]]),
        "ref_words": np.mean([len(str(x).split()) for x in scored["output"]]),
    }])
    return scored, pd.concat([overall, summary], ignore_index=True)

print("Scoring utilities ready.")

Scoring utilities ready.


In [10]:
# Smoke validation: one row per subset.
smoke_df = (
    val_df.sample(frac=1.0, random_state=SEED)
    .groupby("subset", group_keys=False)
    .head(1)
    .sort_values("subset")
    .reset_index(drop=True)
)

SMOKE_PATH = OUTPUT_DIR / "aya_23_8b_zero_shot_smoke_predictions.csv"
smoke_pred_df = run_generation_with_checkpoints(
    smoke_df,
    SMOKE_PATH,
    checkpoint_every=1,
    overwrite=True,
)
smoke_scored_df, smoke_summary_df = score_predictions(smoke_pred_df, smoke_df)
display(smoke_summary_df.round(4))
display(smoke_scored_df[["ID", "subset", "input", "output", "prediction", "rouge1_f1", "rougeL_f1"]])

  0%|          | 0/8 [00:00<?, ?it/s]

Saved 8 predictions to /content/multilingual-health-question-answering/outputs/aya_23_8b_zero_shot_smoke_predictions.csv
Elapsed: 0.9 minutes


,subset,rows,rouge1_f1,rougeL_f1,local_rouge_weighted,pred_words,ref_words
0,OVERALL,8,0.2246,0.1507,0.1389,85.25,97.375
1,Aka_Gha,1,0.3289,0.2500,0.2142,50.00,76.000
2,Amh_Eth,1,0.0000,0.0000,0.0000,13.00,17.000
3,Eng_Eth,1,0.2268,0.1443,0.1373,66.00,30.000
4,Eng_Gha,1,0.3109,0.1969,0.1879,145.00,49.000
5,Eng_Ken,1,0.3833,0.2160,0.2217,172.00,107.000
6,Eng_Uga,1,0.2734,0.1406,0.1532,166.00,337.000
7,Lug_Uga,1,0.1803,0.1803,0.1334,50.00,51.000
8,Swa_Ken,1,0.0930,0.0775,0.0631,20.00,112.000


,ID,subset,input,output,prediction,rouge1_f1,rougeL_f1
0,ID_VL_Aka_Gha_EB2263DD,Aka_Gha,Ɔkwan bɛn so na mmabun betumi akamfo wɔn ho ahwɛ ahu sɛ wobenya ayaresa a obu ne animhwɛ nnim afi akwaahosan ho adwumayɛfo hɔ?,"Ɛsɛ sɛ mmabun di kan kyerɛkyerɛ wɔn ho akwannya ahorow ho asɛm, na wɔkyerɛ hokwan a wɔwɔ sɛ wɔde obu ne animhwɛ nnim di dwuma denam wɔn ahiade ne nea wɔpɛ a wɔbɛka akyerɛ akwahosan ho adwumayɛfo pefee no so. Wobetumi ahwehwɛ akwahosan h...","Ɔkwan bɛn so na mmabun betumi akamfo wɔn ho ahwɛ ahu sɛ wobenya ayaresa aobu ne animhwɛ nnim afi akwaahosan ho adwumyɛfo hɔ, wɔn ne animhwɛ nnim afi akwaahosan ho adwumyɛfo hɔ, wɔn ne animhwɛ nnim afi akwaahosan ho adwumyɛfo hɔ, wɔn ne ...",0.328947,0.250000
1,ID_VL_Amh_Eth_A5CB63A2,Amh_Eth,በሰውነቴ እና በመልክዬ ኩራት ይሰማኛል?,አይከንቱ ወይም ጉራ መሆን ስህተት ነው፣ ነገር ግን የሰውነትዎን ጥንካሬ እና ጤንነት ማድነቅ ለእግዚአብሔር ፍጥረት አክብሮት ያሳያል።,እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እ�,0.000000,0.000000
2,ID_VL_Eng_Eth_01902620,Eng_Eth,"If I get the medicine for Gonorrhea, should my partner also take the medicine, even if he says he feels fine?","This is a question about, Partner Treatment. Yes, always. He may be asymptomatic (no signs), but he will still reinfect you. Both partners must be treated at the same time.","Yes, it is important that both you and your partner receive treatment for gonorrhea to ensure complete eradication of the infection and to prevent reinfection. Gonorrhea can be a silent infection with no noticeable symptoms, especially ...",0.226804,0.144330
3,ID_VL_Eng_Gha_2CD2A404,Eng_Gha,"Are there specific healthcare providers or clinics that specialize in providing medical assistance to adolescents in crisis situations, and how can adolescents find them?","There are specific healthcare providers or clinics that specialize in providing medical assistance to adolescents in crisis situations, such as sexual health clinics, family planning clinics, or adolescent health centers. Adolescents ca...","Yes, there are healthcare providers and clinics that offer specialized services for adolescents in crisis situations. These services are often provided by youth-friendly clinics or organizations that understand the unique needs and chal...",0.310881,0.196891
4,ID_VL_Eng_Ken_0B531932,Eng_Ken,Can I have a child with a HIV patient without getting infected?,"Yes, it is possible for an HIV-negative individual to have a child with an HIV-positive partner without acquiring HIV if certain precautions and medical interventions are followed. The process typically involves a combination of strateg...",It is possible to conceive a child with an HIV-positive partner if both you and your partner take the necessary precautions and follow the recommended guidelines. Here's what you need to know: - Pre-test your partner: Ensure your partne...,0.383275,0.216028
5,ID_VL_Eng_Uga_629EF3D8,Eng_Uga,"How can I protect myself from contracting Herpes Simplex Virus?, please answer in detail.","The only way to completely avoid STIs is to not have vaginal, anal, or oral sex. If you are sexually active, you can do the following things to lower your chances of getting Genital Herpes or Herpes Simplex Virus (HSV), also known as he...","To reduce the risk of contracting Herpes Simplex Virus (HSV), it is important to practice safe sexual practices. Here are some key steps: - Practice safe sex: Use protection, such as condoms, during sexual activity. This significantly r...",0.273438,0.140625
6,ID_VL_Lug_Uga_AD409244,Lug_Uga,Nsobola ntya okuwangaala n'Akawuka akaleeta Siriimu/Siriimu?,"Okuwangaala n'okuweeweeza Akawuka akaleeta Siriimu kisobokera ddala n'obujjanjabi obutuufu, obujjanjabi n'engeri gy'osalawo okweyisa. Abantu abalina Akawuka akaleeta Siriimu basobola okuwangaala nako mu bulamu obweyagaza nga balina obuj...","Okwangwa okuwangaala n'akawuka akaleeta Siriimu. Okukwanga n'okukwanga n'okugula omugisa okuwangaala, okukwanga n'okugula omugisa okuwangaala, okukwanga n'okugula omugisa ok

In [11]:
# Stratified validation sample. Increase SAMPLE_PER_SUBSET when smoke output looks healthy.
SAMPLE_PER_SUBSET = 10
sample_val_df = (
    val_df.sample(frac=1.0, random_state=SEED)
    .groupby("subset", group_keys=False)
    .head(SAMPLE_PER_SUBSET)
    .sort_values(["subset", "ID"])
    .reset_index(drop=True)
)

SAMPLE_VAL_PATH = OUTPUT_DIR / f"aya_23_8b_zero_shot_val_sample_{SAMPLE_PER_SUBSET}_per_subset.csv"
sample_pred_df = run_generation_with_checkpoints(
    sample_val_df,
    SAMPLE_VAL_PATH,
    checkpoint_every=10,
    overwrite=False,
)
sample_scored_df, sample_summary_df = score_predictions(sample_pred_df, sample_val_df)
display(sample_summary_df.round(4))

  0%|          | 0/80 [00:00<?, ?it/s]

Saved 80 predictions to /content/multilingual-health-question-answering/outputs/aya_23_8b_zero_shot_val_sample_10_per_subset.csv
Elapsed: 6.8 minutes


,subset,rows,rouge1_f1,rougeL_f1,local_rouge_weighted,pred_words,ref_words
0,OVERALL,80,0.2128,0.1470,0.1331,67.8625,80.175
1,Aka_Gha,10,0.1921,0.1694,0.1337,60.6000,97.500
2,Amh_Eth,10,0.0000,0.0000,0.0000,17.1000,15.800
3,Eng_Eth,10,0.2575,0.1874,0.1646,63.8000,23.600
4,Eng_Gha,10,0.3766,0.2223,0.2216,130.5000,84.500
5,Eng_Ken,10,0.3007,0.1894,0.1813,90.8000,91.500
6,Eng_Uga,10,0.3231,0.1868,0.1887,116.1000,146.400
7,Lug_Uga,10,0.1304,0.1236,0.0940,18.6000,78.700
8,Swa_Ken,10,0.1220,0.0975,0.0812,45.4000,103.400


In [12]:
# Full validation is optional and can be slow.
RUN_FULL_VALIDATION = False

FULL_VAL_PATH = OUTPUT_DIR / "aya_23_8b_zero_shot_val_predictions.csv"
FULL_VAL_SCORED_PATH = OUTPUT_DIR / "aya_23_8b_zero_shot_val_scored.csv"
FULL_VAL_SUMMARY_PATH = OUTPUT_DIR / "aya_23_8b_zero_shot_val_summary.csv"

if RUN_FULL_VALIDATION:
    full_val_pred_df = run_generation_with_checkpoints(
        val_df,
        FULL_VAL_PATH,
        checkpoint_every=25,
        overwrite=False,
    )
    full_val_scored_df, full_val_summary_df = score_predictions(full_val_pred_df, val_df)
    full_val_scored_df.to_csv(FULL_VAL_SCORED_PATH, index=False)
    full_val_summary_df.to_csv(FULL_VAL_SUMMARY_PATH, index=False)
    display(full_val_summary_df.round(4))
else:
    print("RUN_FULL_VALIDATION is False. Using sample_scored_df for error analysis.")
    full_val_scored_df = sample_scored_df.copy()
    full_val_summary_df = sample_summary_df.copy()

RUN_FULL_VALIDATION is False. Using sample_scored_df for error analysis.


In [13]:
def add_error_flags(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()
    out["prediction"] = out["prediction"].fillna("").astype(str)
    out["pred_words"] = out["prediction"].str.split().str.len()
    out["ref_words"] = out["output"].fillna("").astype(str).str.split().str.len()
    out["is_empty"] = out["prediction"].str.strip().eq("")
    out["maybe_too_short"] = out["pred_words"] < 0.35 * out["ref_words"].clip(lower=1)
    out["maybe_too_long"] = out["pred_words"] > 2.5 * out["ref_words"].clip(lower=1)
    out["contains_refusal"] = out["prediction"].str.contains(
        r"cannot|can't|unable|consult a doctor|medical professional|as an ai",
        case=False,
        regex=True,
    )
    return out

analysis_df = add_error_flags(full_val_scored_df)
display(analysis_df[["is_empty", "maybe_too_short", "maybe_too_long", "contains_refusal"]].mean().to_frame("rate"))
lowest_cols = ["ID", "subset", "input", "output", "prediction", "rouge1_f1", "rougeL_f1", "pred_words", "ref_words"]
display(analysis_df.sort_values("rougeL_f1").head(10)[lowest_cols])

,rate
is_empty,0.0000
maybe_too_short,0.2125
maybe_too_long,0.1625
contains_refusal,0.0250


,ID,subset,input,output,prediction,rouge1_f1,rougeL_f1,pred_words,ref_words
12,ID_VL_Amh_Eth_4DD8EB85,Amh_Eth,ገና 14 ዓመቴ ነው፣ እና ክሊኒኩ ትልቅ ሰው ማምጣት እንዳለብኝ ነገረኝ። አልችልም፣ ምን ማድረግ ይገባኛል?,ማህበራዊ ሰራተኛ ወይም ልዩ የጤና ሰራተኛ ይጠይቁ። ለአካለ መጠን ያልደረሱ ልጆች ያለ ቤተሰባቸው ሚስጥራዊ እንክብካቤ እንዲያገኙ ለመርዳት የሰለጠኑ ናቸው።,እንዳለብኝ እንገልግግሎት እንገልግግሎት እንገልግግሎት እንገልግግሎት እንገልግግሎት እንገልግግሎት እንገልግግሎት እንገልግግሎት እንገልግግሎት እንገልግ�,0.0,0.0,11,19
13,ID_VL_Amh_Eth_670ACC5C,Amh_Eth,ለተጨማሪ ደህንነት ሁለት ኮንዶም በአንድ ጊዜ (አንዱ በሌላው ላይ) መጠቀም ደህንነቱ የተጠበቀ ነው?,አይሁለት ኮንዶም መጠቀም አንዱን ከመጠቀም ያነሰ ደህንነቱ የተጠበቀ ነው፣ ምክንያቱም በንብርብሮች መካከል ያለው ግጭት ሁለቱንም በቀላሉ እንዲቀደዱ ሊያደርግ ይችላል። ሁል ጊዜ ነጠላ ኮንዶም በትክክል ይጠቀሙ።,እናት አንድ ጊዜ ምርካ ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክን ልንክ�,0.0,0.0,20,25
15,ID_VL_Amh_Eth_7A8D03AA,Amh_Eth,ሄርፒስ እንዴት ይተላለፋል?,ሄርፒስ ምንም አይነት ምልክት ባይኖርዉም እንኳ የሚተላለፈው ቆዳ ለቆዳ በመነካካት፣ በመሳሳም ወይም በጾታዊ ግንኙነት ነው።,እንዴት ይተላለፋል የአገልግሎት ባልነት የአገልግሎት ባልነት የአገልግሎት ባልነት የአገልግሎት ባልነት የአገልግሎት ባልነት የአገልግሎት ባልነት የአገልግሎት ባልነት,0.0,0.0,16,15
14,ID_VL_Amh_Eth_6A5D932B,Amh_Eth,ትሪኮሞኒያሲስ የሚያመጣው ምን ዓይነት አካል ነው?,ነጠላ-ሕዋስ ፕሮቶዞአን ጥገኛ ተውሳክ (Trichomonas vaginalis)፡፡,እናት አካል አንግልን ዘብንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውንን ውን�,0.0,0.0,24,6
10,ID_VL_Amh_Eth_0DF56AC6,Amh_Eth,HPV በወንዶች ላይ ያጠቃል እና ሊድንስ ይችላል?,አዎHPV ወንዶችንም ሴቶችንም ያጠቃል እና በወንዶች ላይ ካንሰር እና የብልት ኪንታሮት ሊያስከትል ይችላል::,እንነቱ የእናት ምርናት ልንክን የእናት ምርናት ልንክን የእናት ምርናት ልንክን የእናት ምርናት ልንክን የእናት ምርናት ልንክን የእናት ምርናት ልንክን የእናት,0.0,0.0,20,13
11,ID_VL_Amh_Eth_409D3602,Amh_Eth,በፕሮቶዞአን ጥገኛ ተውሳክ የሚከሰተው የትኛው ዓይነት የአባላዘር በሽታ ነው?,ትሪኮሞኒየስ (Trichomonas vaginalis)፡፡,እንገም የአባላዘር በሽታ የአራእን የእናት ዓይነት የአራእን የእናት ዓይነት የአራእን የእናት ዓይነት የአራእን የእናት ዓይነት የአራእን የእናት ዓይነት የአራእ�,0.0,0.0,19,3
17,ID_VL_Amh_Eth_A5CB63A2,Amh_Eth,በሰውነቴ እና በመልክዬ ኩራት ይሰማኛል?,አይከንቱ ወይም ጉራ መሆን ስህተት ነው፣ ነገር ግን የሰውነትዎን ጥንካሬ እና ጤንነት ማድነቅ ለእግዚአብሔር ፍጥረት አክብሮት ያሳያል።,እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እንገልግሎት እ�,0.0,0.0,13,17
16,ID_VL_Amh_Eth_9DAC6E83,Amh_Eth,ትሪኮሞኒያሲስ ለኤችአይቪ ተጋላጭነትን ይጨምራል?,አዎ። ትሪኮሞኒያሲስ ካለብዎት እና ለኤችአይቪ ከተጋለጡ በቫይረሱ የመያዝ ዕድልዎን ይበልጥ ከፍተኛ ያደርገዋል።,እናት እንገልግሎት ዘባእንገልግሎት ዘባእንገልግሎት ዘባእንገልግሎት ዘባእንገልግሎት ዘባእንገልግሎት ዘባእንገልግሎት ዘባእንገልግሎት ዘባእንገልግሎት �,0.0,0.0,11,12
19,ID_VL_Amh_Eth_B04D3376,Amh_Eth,አንድ ሕፃን ከተወለደ በኋላ ወዲያውኑ የዓይን ሕመም ቢይዝ ይህ ሕመም ከእናቱ የተላለፈ ሊሆን ይችላል?,አዎ፣ ሊሆን ይችላል። በሕፃናት ላይ የሚከሰት የአይን ኢንፌክሽኖች (neonatal conjunctivitis)በዋነኝነት ምክንያቱ በወሊድ ጊዜ በሚተላለፍ ክላሚዲያ ወይም ጨብጥ ነው። ዓይነ ስውርነትን ለመከላከል ወዲያውኑ መታከም አለበት።,እናቱ የተላለፈ ሊሆን ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል ይኽላል �,0.0,0.0,23,24
18,ID_VL_Amh_Eth_AB8DDEB4,Amh_Eth,በጤና ጣቢያው ውስጥ ለነፍሰ ጡር ታዳጊ ልጃገረዶች ምን ዓይነት አገልግሎቶች ነፃ ናቸው?,የቅድመ ወሊድ እንክብካቤ ምርመራዎች፣ የቴታነስ ክትባቶች፣ የብረት/ፎሊክ አሲድ ተጨማሪዎች እና ልጅ መውለድ ብዙ ጊዜ ነፃ ወይም ዝቅተኛ ዋጋ አላቸው። ለበለጠ መረጃ የአካባቢዎን ክሊኒክ ያነጋግሩ።,እናት አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገልግሎቶች አገል,0.0,0.0,14,24


## 5. Test Prediction File

Run test inference only after the smoke and validation sample look competitive. The same answer is duplicated across all target columns.

In [14]:
RUN_TEST_INFERENCE = False

TEST_PRED_PATH = OUTPUT_DIR / "aya_23_8b_zero_shot_test_predictions.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submission_aya_23_8b_zero_shot.csv"

if RUN_TEST_INFERENCE:
    test_pred_df = run_generation_with_checkpoints(
        test_df,
        TEST_PRED_PATH,
        checkpoint_every=25,
        overwrite=False,
    )
else:
    print("RUN_TEST_INFERENCE is False.")
    if TEST_PRED_PATH.exists():
        test_pred_df = pd.read_csv(TEST_PRED_PATH)
        print("Loaded existing test predictions:", TEST_PRED_PATH)
    else:
        test_pred_df = None
        print("No existing test predictions found.")

RUN_TEST_INFERENCE is False.
No existing test predictions found.


In [15]:
def create_prediction_file(test_predictions: pd.DataFrame, sample_submission: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    if test_predictions is None or test_predictions.empty:
        raise ValueError("test_predictions is empty. Run test inference first.")

    missing = {"ID", "prediction"} - set(test_predictions.columns)
    if missing:
        raise ValueError(f"test_predictions is missing columns: {sorted(missing)}")

    pred_map = dict(zip(test_predictions["ID"], test_predictions["prediction"].fillna("")))
    missing_ids = [row_id for row_id in sample_submission["ID"] if row_id not in pred_map]
    if missing_ids:
        raise ValueError(f"Missing predictions for {len(missing_ids)} IDs. First few: {missing_ids[:5]}")

    answers = [clean_generation(pred_map[row_id]) for row_id in sample_submission["ID"]]
    if any(not ans for ans in answers):
        raise ValueError("Found empty predictions. Fix before creating the prediction file.")

    prediction_file = pd.DataFrame({
        "ID": sample_submission["ID"],
        "TargetRLF1": answers,
        "TargetR1F1": answers,
        "TargetLLM": answers,
    })

    expected_cols = ["ID", "TargetRLF1", "TargetR1F1", "TargetLLM"]
    assert prediction_file.columns.tolist() == expected_cols
    assert len(prediction_file) == len(sample_submission)
    assert prediction_file["TargetRLF1"].equals(prediction_file["TargetR1F1"])
    assert prediction_file["TargetRLF1"].equals(prediction_file["TargetLLM"])
    assert not prediction_file.isna().any().any()

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    prediction_file.to_csv(output_path, index=False)
    print("Saved prediction file:", output_path)
    return prediction_file

if test_pred_df is not None:
    prediction_file_df = create_prediction_file(test_pred_df, sample_sub_df, SUBMISSION_PATH)
    display(prediction_file_df.head())
else:
    print("Prediction file not created because test_pred_df is None.")

Prediction file not created because test_pred_df is None.


## Run Notes

Record these values after each run:

- Model: `CohereForAI/aya-23-8B`
- Quantization: none / `USE_4BIT = False`
- Validation sample size
- ROUGE-1 F1 and ROUGE-L F1
- Public leaderboard metrics after upload
- Runtime and peak GPU memory

Compare against the current Gemma 4 E4B zero-shot baseline before running a full test inference.